@torch.no_grad() 是PyTorch中一个上下文管理器装饰器，用于禁用梯度计算，从而节省内存和计算资源。

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# 核心作用

##  禁用梯度计算
在 torch.no_grad() 块内，所有计算都不会被跟踪，也不会构建计算图（computation graph），因此不会计算梯度。
这意味着：
- requires_grad 会被临时忽略。
- .grad 属性不会被填充。
- 反向传播（backward()）无法在该块内执行。

## 2. 节省内存

PyTorch 在前向传播时会缓存中间变量以用于反向传播。如果禁用梯度，这些缓存就不会被创建，从而显著减少 GPU/CPU 内存占用。

## 3. 加速计算

由于不需要构建计算图和计算梯度，运算速度会更快，尤其在推理（inference）阶段非常有用。

## 典型使用场景

### ✅ 场景 1：模型推理（Inference）

在模型训练完成后，进行预测时不需要梯度。

In [ ]:
model.eval()
with torch.no_grad():
    outputs = model(inputs)
    # 不需要backward，也不需要计算梯度

### ✅ 场景 2：评估/验证（Validation/Test）
在验证或测试阶段，通常不需要更新参数，因此关闭梯度可以节省资源。

In [ ]:
model.eval()
with torch.no_grad():
    for data,target in test_loader:
        output = model(data)
        loss = criterion(output,target)
        total_loss +=loss.item()

### ✅ 场景 3：冻结参数训练（Fine-tuning）
当你只想训练模型的一部分（如只训练分类头，冻结 backbone）时，可以用 torch.no_grad() 包裹不需要更新的部分。

In [ ]:
# 冻结backbone
with torch.no_grad():
    features = backbone(images)
    
# 只训练classifier
output = classifier(featrues)
loss = criterion(outputs,labels)
loss.backward() # 只有classifier的参数会被更新

In [4]:


def why_freeze_parameters():
    """
    解释为什么需要冻结参数
    """
    print("=== 为什么需要冻结参数 ===\n")
    
    reasons = [
        "1. 防止过拟合：预训练模型已经学到了通用特征",
        "2. 节省显存：冻结的参数不需要存储梯度和优化器状态",
        "3. 加速训练：减少需要计算的梯度数量",
        "4. 保持预训练知识：避免破坏已经学到的有用特征"
    ]
    
    for reason in reasons:
        print(reason)
    
    # 显存对比
    print("\n显存占用对比:")
    model = nn.Sequential(
        nn.Linear(1000, 500),
        nn.ReLU(),
        nn.Linear(500, 100),
        nn.ReLU(),
        nn.Linear(100, 10)
    )
    
    # 计算参数量
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = total_params
    
    print(f"  全部参数可训练: {total_params/1e6:.2f}M 参数")
    print(f"  梯度显存占用: {total_params * 4 / 1024**2:.2f} MB (fp32)")
    
    # 冻结部分层
    for param in model[0].parameters():
        param.requires_grad = False
    
    frozen_params = sum(p.numel() for p in model[0].parameters())
    trainable_params = total_params - frozen_params
    
    print(f"\n  冻结第一层后:")
    print(f"    冻结参数: {frozen_params/1e6:.2f}M")
    print(f"    可训练参数: {trainable_params/1e6:.2f}M")
    print(f"    节省显存: {frozen_params * 4 / 1024**2:.2f} MB")

why_freeze_parameters()

=== 为什么需要冻结参数 ===

1. 防止过拟合：预训练模型已经学到了通用特征
2. 节省显存：冻结的参数不需要存储梯度和优化器状态
3. 加速训练：减少需要计算的梯度数量
4. 保持预训练知识：避免破坏已经学到的有用特征

显存占用对比:
  全部参数可训练: 0.55M 参数
  梯度显存占用: 2.10 MB (fp32)

  冻结第一层后:
    冻结参数: 0.50M
    可训练参数: 0.05M
    节省显存: 1.91 MB


### 冻结特定层

In [5]:
def freeze_with_no_grad():
    """
    使用 torch.no_grad() 冻结参数
    """
    print("=== 方法1: 使用 torch.no_grad() 冻结参数 ===\n")
    
    # 创建预训练模型
    class PretrainedModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = nn.Sequential(
                nn.Linear(784, 256),
                nn.ReLU(),
                nn.Linear(256, 128),
                nn.ReLU()
            )
            self.classifier = nn.Linear(128, 10)
        
        def forward(self, x):
            features = self.backbone(x)
            return self.classifier(features)
    
    model = PretrainedModel()
    
    # 方法1：在 forward 中使用 no_grad 冻结 backbone
    class FrozenBackboneModel(nn.Module):
        def __init__(self, pretrained_model):
            super().__init__()
            self.backbone = pretrained_model.backbone
            self.classifier = pretrained_model.classifier
        
        def forward(self, x):
            # ✅ 使用 no_grad 冻结 backbone
            with torch.no_grad():
                features = self.backbone(x)
            # 分类器正常计算梯度
            return self.classifier(features)
    
    frozen_model = FrozenBackboneModel(model)
    
    # 检查哪些参数需要梯度
    print("参数梯度状态:")
    for name, param in frozen_model.named_parameters():
        print(f"  {name}: requires_grad={param.requires_grad}")
    
    # 模拟训练
    x = torch.randn(32, 784)
    y = torch.randint(0, 10, (32,))
    
    optimizer = optim.Adam(frozen_model.classifier.parameters(), lr=0.001)
    
    print("\n训练过程:")
    for epoch in range(3):
        optimizer.zero_grad()
        output = frozen_model(x)
        loss = F.cross_entropy(output, y)
        loss.backward()
        optimizer.step()
        
        print(f"  Epoch {epoch+1}: loss={loss.item():.4f}")
        
        # 验证 backbone 参数没有梯度
        backbone_grad = any(p.grad is not None for p in frozen_model.backbone.parameters())
        print(f"    backbone 有梯度: {backbone_grad}")

freeze_with_no_grad()

=== 方法1: 使用 torch.no_grad() 冻结参数 ===

参数梯度状态:
  backbone.0.weight: requires_grad=True
  backbone.0.bias: requires_grad=True
  backbone.2.weight: requires_grad=True
  backbone.2.bias: requires_grad=True
  classifier.weight: requires_grad=True
  classifier.bias: requires_grad=True

训练过程:
  Epoch 1: loss=2.2778
    backbone 有梯度: False
  Epoch 2: loss=2.2693
    backbone 有梯度: False
  Epoch 3: loss=2.2609
    backbone 有梯度: False


## 选择性冻结：在训练循环中动态控制

In [ ]:
def selective_freeze():
    """
    在训练循环中动态选择冻结哪些层
    """
    print("=== 方法2: 动态选择冻结层 ===\n")
    
    class FlexibleModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(3, 64, 3)
            self.conv2 = nn.Conv2d(64, 128, 3)
            self.conv3 = nn.Conv2d(128, 256, 3)
            self.fc = nn.Linear(256 * 26 * 26, 10)
        
        def forward(self, x, freeze_layers=None):
            if freeze_layers is None:
                freeze_layers = []
            
            # 动态冻结指定层
            x = self.conv1(x)
            if 'conv1' in freeze_layers:
                x = x.detach()  # 断开梯度
            
            x = F.relu(x)
            x = self.conv2(x)
            if 'conv2' in freeze_layers:
                x = x.detach()
            
            x = F.relu(x)
            x = self.conv3(x)
            if 'conv3' in freeze_layers:
                x = x.detach()
            
            x = F.relu(x)
            x = x.view(x.size(0), -1)
            return self.fc(x)
    
    model = FlexibleModel()
    x = torch.randn(32, 3, 224, 224)
    y = torch.randint(0, 10, (32,))
    
    print("训练策略:")
    strategies = [
        ("全部训练", []),
        ("冻结 conv1", ['conv1']),
        ("冻结 conv1, conv2", ['conv1', 'conv2']),
    ]
    
    for strategy_name, freeze_layers in strategies:
        print(f"\n{strategy_name}:")
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        
        for epoch in range(2):
            optimizer.zero_grad()
            output = model(x, freeze_layers=freeze_layers)
            loss = F.cross_entropy(output, y)
            loss.backward()
            optimizer.step()
            
            # 检查各层梯度
            conv1_grad = model.conv1.weight.grad is not None
            conv2_grad = model.conv2.weight.grad is not None
            conv3_grad = model.conv3.weight.grad is not None
            
            print(f"  Epoch {epoch+1}: loss={loss.item():.4f}")
            print(f"    conv1梯度: {conv1_grad}, conv2梯度: {conv2_grad}, conv3梯度: {conv3_grad}")

selective_freeze()

=== 方法2: 动态选择冻结层 ===

训练策略:

全部训练:


## 永久冻结参数

In [ ]:
def freeze_with_requires_grad():
    """
    使用 requires_grad = False 永久冻结参数
    """
    print("=== 方法3: 使用 requires_grad = False 冻结参数 ===\n")
    
    # 加载预训练模型
    class ResNetLike(nn.Module):
        def __init__(self):
            super().__init__()
            self.conv1 = nn.Conv2d(3, 64, 7, stride=2)
            self.bn1 = nn.BatchNorm2d(64)
            self.layer1 = nn.Sequential(
                nn.Conv2d(64, 64, 3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU()
            )
            self.layer2 = nn.Sequential(
                nn.Conv2d(64, 128, 3, stride=2, padding=1),
                nn.BatchNorm2d(128),
                nn.ReLU()
            )
            self.fc = nn.Linear(128 * 56 * 56, 1000)
        
        def forward(self, x):
            x = F.relu(self.bn1(self.conv1(x)))
            x = self.layer1(x)
            x = self.layer2(x)
            x = x.view(x.size(0), -1)
            return self.fc(x)
    
    model = ResNetLike()
    
    print("初始状态:")
    print(f"  总参数: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
    print(f"  可训练参数: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")
    
    # 冻结卷积层
    for name, param in model.named_parameters():
        if 'conv' in name or 'bn' in name:
            param.requires_grad = False
    
    print("\n冻结卷积层和BN层后:")
    print(f"  可训练参数: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M")
    
    # 只训练分类头
    optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
    
    # 验证梯度
    x = torch.randn(32, 3, 224, 224)
    y = torch.randint(0, 1000, (32,))
    
    output = model(x)
    loss = F.cross_entropy(output, y)
    loss.backward()
    
    print("\n梯度状态:")
    print(f"  conv1 有梯度: {model.conv1.weight.grad is not None}")
    print(f"  layer1 有梯度: {model.layer1[0].weight.grad is not None}")
    print(f"  fc 有梯度: {model.fc.weight.grad is not None}")

freeze_with_requires_grad()

## 渐进式解冻（Gradual Unfreezing）

In [2]:
def gradual_unfreezing():
    """
    渐进式解冻：先训练分类头，逐步解冻更多层
    """
    print("=== 方法4: 渐进式解冻 ===\n")
    
    class GradualUnfreezingModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.ModuleList([
                nn.Linear(100, 200),
                nn.ReLU(),
                nn.Linear(200, 200),
                nn.ReLU(),
                nn.Linear(200, 100),
                nn.ReLU(),
                nn.Linear(100, 10)
            ])
        
        def forward(self, x):
            for layer in self.layers:
                x = layer(x)
            return x
    
    model = GradualUnfreezingModel()
    
    # 初始：冻结所有层
    for param in model.parameters():
        param.requires_grad = False
    
    # 逐层解冻
    unfreeze_schedule = [
        ("第1阶段: 只训练最后一层", [-1]),  # 只训练最后一层
        ("第2阶段: 训练最后3层", [-1, -2, -3]),  # 训练最后3层
        ("第3阶段: 训练所有层", None)  # 训练所有层
    ]
    
    x = torch.randn(32, 100)
    y = torch.randint(0, 10, (32,))
    
    for stage_name, layers_to_unfreeze in unfreeze_schedule:
        print(f"\n{stage_name}")
        
        # 解冻指定层
        if layers_to_unfreeze is not None:
            for idx in layers_to_unfreeze:
                for param in model.layers[idx].parameters():
                    param.requires_grad = True
        else:
            for param in model.parameters():
                param.requires_grad = True
        
        # 统计可训练参数
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        total = sum(p.numel() for p in model.parameters())
        print(f"  可训练参数: {trainable/1e3:.1f}K / {total/1e3:.1f}K ({trainable/total*100:.1f}%)")
        
        # 训练几个epoch
        optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)
        
        for epoch in range(2):
            optimizer.zero_grad()
            output = model(x)
            loss = F.cross_entropy(output, y)
            loss.backward()
            optimizer.step()
            print(f"    Epoch {epoch+1}: loss={loss.item():.4f}")

gradual_unfreezing()

=== 方法4: 渐进式解冻 ===


第1阶段: 只训练最后一层
  可训练参数: 1.0K / 81.5K (1.2%)
    Epoch 1: loss=2.2902
    Epoch 2: loss=2.2880

第2阶段: 训练最后3层
  可训练参数: 21.1K / 81.5K (25.9%)
    Epoch 1: loss=2.2857
    Epoch 2: loss=2.2713

第3阶段: 训练所有层
  可训练参数: 81.5K / 81.5K (100.0%)
    Epoch 1: loss=2.2578
    Epoch 2: loss=2.2158


| 功能 | 作用 | 是否影响 `requires_grad` | 是否构建计算图 |
|------|------|------------------------|----------------|
| `torch.no_grad()` | 禁用梯度计算 | 否（但临时忽略） | 否 |
| `requires_grad = False` | 禁止某个张量计算梯度 | 是（永久设置） | 否 |
| `torch.set_grad_enabled(bool)` | 动态开启/关闭梯度 | 否 | 根据布尔值决定 |
| `torch.no_grad()` 装饰器 | 作用于函数 | 否 | 否 |



## 最佳实践总结

In [1]:
def best_practices():
    """
    冻结参数训练的最佳实践
    """
    print("=== 最佳实践总结 ===\n")
    
    practices = {
        "✅ 推荐做法": [
            "使用 requires_grad=False 永久冻结参数",
            "使用 with torch.no_grad() 临时冻结前向传播",
            "渐进式解冻：先训练头部，再逐步解冻",
            "使用不同的学习率：头部用大学习率，backbone用小学习率"
        ],
        "❌ 避免做法": [
            "频繁切换 requires_grad 状态",
            "在训练循环中创建新的 no_grad 上下文（性能开销）",
            "忘记设置模型为 eval 模式（影响 Dropout/BatchNorm）",
            "冻结所有层导致欠拟合"
        ]
    }
    
    for category, items in practices.items():
        print(f"{category}:")
        for item in items:
            print(f"  • {item}")
    
    print("\n完整微调模板:")
    print("""
    # 1. 加载预训练模型
    model = torchvision.models.resnet50(pretrained=True)
    
    # 2. 冻结 backbone
    for param in model.parameters():
        param.requires_grad = False
    
    # 3. 替换分类头
    num_classes = 10
    model.fc = nn.Linear(2048, num_classes)
    
    # 4. 创建优化器（只优化新层）
    optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
    
    # 5. 训练
    model.train()
    for epoch in range(num_epochs):
        for batch in dataloader:
            optimizer.zero_grad()
            output = model(batch[0])
            loss = criterion(output, batch[1])
            loss.backward()
            optimizer.step()
    
    # 6. 可选：渐进式解冻
    for param in model.layer4.parameters():
        param.requires_grad = True
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)
    """)

best_practices()

=== 最佳实践总结 ===

✅ 推荐做法:
  • 使用 requires_grad=False 永久冻结参数
  • 使用 with torch.no_grad() 临时冻结前向传播
  • 渐进式解冻：先训练头部，再逐步解冻
  • 使用不同的学习率：头部用大学习率，backbone用小学习率
❌ 避免做法:
  • 频繁切换 requires_grad 状态
  • 在训练循环中创建新的 no_grad 上下文（性能开销）
  • 忘记设置模型为 eval 模式（影响 Dropout/BatchNorm）
  • 冻结所有层导致欠拟合

完整微调模板:

    # 1. 加载预训练模型
    model = torchvision.models.resnet50(pretrained=True)
    
    # 2. 冻结 backbone
    for param in model.parameters():
        param.requires_grad = False
    
    # 3. 替换分类头
    num_classes = 10
    model.fc = nn.Linear(2048, num_classes)
    
    # 4. 创建优化器（只优化新层）
    optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
    
    # 5. 训练
    model.train()
    for epoch in range(num_epochs):
        for batch in dataloader:
            optimizer.zero_grad()
            output = model(batch[0])
            loss = criterion(output, batch[1])
            loss.backward()
            optimizer.step()
    
    # 6. 可选：渐进式解冻
    for param in model.layer4.parameters():
        param.requires_

In [ ]:
import torch

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x * 2

with torch.no_grad():
    z = y * 3  # z 的计算不会记录到计算图中，也不会计算梯度

print(z.requires_grad)  # 输出: False

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [1]:


def basic_usage():
    """
    @torch.no_grad() 的基本作用
    """
    print("=== @torch.no_grad() 基本作用 ===\n")
    
    # 创建需要梯度的张量
    x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
    print(f"x: {x}, requires_grad={x.requires_grad}")
    
    # 1. 正常计算（会记录计算图）
    y = x ** 2
    print(f"\n正常计算:")
    print(f"  y: {y}, requires_grad={y.requires_grad}")
    print(f"  y.grad_fn: {y.grad_fn}")  # 有梯度函数
    
    # 2. 使用 torch.no_grad() 上下文管理器
    with torch.no_grad():
        z = x ** 2
        print(f"\nwith torch.no_grad():")
        print(f"  z: {z}, requires_grad={z.requires_grad}")
        print(f"  z.grad_fn: {z.grad_fn}")  # 无梯度函数
    
    # 3. 使用装饰器
    @torch.no_grad()
    def compute_square(tensor):
        return tensor ** 2
    
    w = compute_square(x)
    print(f"\n@torch.no_grad() 装饰器:")
    print(f"  w: {w}, requires_grad={w.requires_grad}")
    print(f"  w.grad_fn: {w.grad_fn}")

basic_usage()

=== @torch.no_grad() 基本作用 ===

x: tensor([1., 2., 3.], requires_grad=True), requires_grad=True

正常计算:
  y: tensor([1., 4., 9.], grad_fn=<PowBackward0>), requires_grad=True
  y.grad_fn: <PowBackward0 object at 0x2cf7a9780>

with torch.no_grad():
  z: tensor([1., 4., 9.]), requires_grad=False
  z.grad_fn: None

@torch.no_grad() 装饰器:
  w: tensor([1., 4., 9.]), requires_grad=False
  w.grad_fn: None


In [2]:
def computation_graph_comparison():
    """
    对比有无 no_grad 的计算图构建
    """
    print("=== 计算图构建对比 ===\n")
    
    # 创建输入
    x = torch.tensor([2.0], requires_grad=True)
    
    # 场景1：正常模式（构建计算图）
    print("1. 正常模式（构建计算图）:")
    a = x ** 2          # 记录操作
    b = a + 3           # 记录操作
    c = b * 2           # 记录操作
    print(f"   c: {c.item()}, requires_grad={c.requires_grad}")
    print(f"   计算图: {c.grad_fn}")
    print(f"   中间节点数量: {len(list(c.grad_fn.next_functions))}")
    
    # 反向传播
    c.backward()
    print(f"   x.grad: {x.grad.item()}")  # 梯度存在
    
    # 重置梯度
    x.grad = None
    
    # 场景2：no_grad模式（不构建计算图）
    print("\n2. no_grad模式（不构建计算图）:")
    with torch.no_grad():
        a = x ** 2
        b = a + 3
        c = b * 2
        print(f"   c: {c.item()}, requires_grad={c.requires_grad}")
        print(f"   计算图: {c.grad_fn}")  # None
    
    # 尝试反向传播（会失败）
    try:
        c.backward()
    except RuntimeError as e:
        print(f"   反向传播错误: {e}")
    
    # 场景3：混合模式
    print("\n3. 混合模式（部分禁用梯度）:")
    with torch.no_grad():
        a = x ** 2          # 无梯度
    b = a + 3               # 有梯度（但a无梯度，所以b也无梯度）
    c = b * 2
    print(f"   a.requires_grad: {a.requires_grad}")
    print(f"   c.requires_grad: {c.requires_grad}")
    print(f"   c.grad_fn: {c.grad_fn}")  # None

computation_graph_comparison()

=== 计算图构建对比 ===

1. 正常模式（构建计算图）:
   c: 14.0, requires_grad=True
   计算图: <MulBackward0 object at 0x2cf7a9780>
   中间节点数量: 2
   x.grad: 8.0

2. no_grad模式（不构建计算图）:
   c: 14.0, requires_grad=False
   计算图: None
   反向传播错误: element 0 of tensors does not require grad and does not have a grad_fn

3. 混合模式（部分禁用梯度）:
   a.requires_grad: False
   c.requires_grad: False
   c.grad_fn: None


# 主要使用场景
## 模型评估/推理

In [3]:
def inference_mode():
    """
    模型推理时使用 no_grad
    """
    print("=== 模型推理优化 ===\n")
    
    # 创建模型
    model = nn.Sequential(
        nn.Linear(100, 500),
        nn.ReLU(),
        nn.Linear(500, 500),
        nn.ReLU(),
        nn.Linear(500, 10)
    )
    
    # 创建输入
    x = torch.randn(64, 100)
    
    # 正常推理（会构建计算图，浪费内存）
    print("1. 正常推理（有计算图）:")
    import tracemalloc
    
    tracemalloc.start()
    output1 = model(x)
    current, peak1 = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    print(f"   输出: {output1.shape}")
    print(f"   峰值内存: {peak1 / 1024**2:.2f} MB")
    print(f"   输出requires_grad: {output1.requires_grad}")
    
    # 使用 no_grad 推理（无计算图）
    print("\n2. no_grad推理（无计算图）:")
    tracemalloc.start()
    with torch.no_grad():
        output2 = model(x)
    current, peak2 = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    print(f"   输出: {output2.shape}")
    print(f"   峰值内存: {peak2 / 1024**2:.2f} MB")
    print(f"   输出requires_grad: {output2.requires_grad}")
    
    print(f"\n内存节省: {(peak1 - peak2) / 1024**2:.2f} MB")
    
    # 验证输出一致
    print(f"输出一致: {torch.allclose(output1, output2)}")

inference_mode()

=== 模型推理优化 ===

1. 正常推理（有计算图）:
   输出: torch.Size([64, 10])
   峰值内存: 0.01 MB
   输出requires_grad: True

2. no_grad推理（无计算图）:
   输出: torch.Size([64, 10])
   峰值内存: 0.00 MB
   输出requires_grad: False

内存节省: 0.01 MB
输出一致: True


## 参数更新（优化器内部）

In [3]:
def optimizer_step():
    """
    优化器step内部使用 no_grad
    """
    print("=== 优化器参数更新 ===\n")
    
    # 模拟优化器的step方法
    class SimpleOptimizer:
        def __init__(self, params, lr=0.01):
            self.params = list(params)
            self.lr = lr
        
        @torch.no_grad()  # 关键：参数更新不需要梯度
        def step(self):
            for param in self.params:
                if param.grad is not None:
                    # 参数更新（在no_grad下进行）
                    param.data -= self.lr * param.grad.data
        
        def zero_grad(self):
            for param in self.params:
                if param.grad is not None:
                    param.grad.zero_()
    
    # 演示
    param = torch.tensor([1.0], requires_grad=True)
    optimizer = SimpleOptimizer([param])
    
    print(f"初始参数: {param.item()}")
    
    # 模拟训练步骤
    loss = (param - 2) ** 2  # 目标值是2
    loss.backward()
    print(f"梯度: {param.grad.item()}")
    
    # 更新参数（内部使用 no_grad）
    optimizer.step()
    print(f"更新后参数: {param.item()}")
    print(f"参数requires_grad: {param.requires_grad}")  # 仍然为True
    
    # 验证参数更新不记录梯度
    param.grad = None
    with torch.no_grad():
        param.data = param * 2  # 直接修改data
    
    print(f"\n直接修改data后: {param.item()}")
    print(f"requires_grad: {param.requires_grad}")  # 仍然为True
    print(f"grad_fn: {param.grad_fn}")  # None

optimizer_step()

=== 优化器参数更新 ===

初始参数: 1.0
梯度: -2.0
更新后参数: 1.0199999809265137
参数requires_grad: True

直接修改data后: 2.0399999618530273
requires_grad: True
grad_fn: None


## 梯度检查点中的使用

In [4]:
def gradient_checkpointing():
    """
    梯度检查点中使用 no_grad
    """
    print("=== 梯度检查点优化 ===\n")
    
    def checkpoint_forward(module, *inputs):
        """
        模拟梯度检查点：前向传播不保存中间结果
        """
        with torch.no_grad():
            # 前向传播不记录梯度
            outputs = module(*inputs)
        return outputs
    
    class LargeModule(nn.Module):
        def __init__(self):
            super().__init__()
            self.linear = nn.Linear(1000, 1000)
        
        def forward(self, x):
            return self.linear(x)
    
    module = LargeModule()
    x = torch.randn(100, 1000, requires_grad=True)
    
    print("标准前向传播（记录所有中间结果）:")
    output1 = module(x)
    print(f"  输出requires_grad: {output1.requires_grad}")
    print(f"  计算图节点: {output1.grad_fn}")
    
    print("\n检查点前向传播（不记录中间结果）:")
    output2 = checkpoint_forward(module, x)
    print(f"  输出requires_grad: {output2.requires_grad}")
    print(f"  计算图节点: {output2.grad_fn}")
    
    # 反向传播需要重新计算
    print("\n检查点反向传播需要重新计算前向结果")

gradient_checkpointing()

=== 梯度检查点优化 ===

标准前向传播（记录所有中间结果）:
  输出requires_grad: True
  计算图节点: <AddmmBackward0 object at 0x2d07bac80>

检查点前向传播（不记录中间结果）:
  输出requires_grad: False
  计算图节点: None

检查点反向传播需要重新计算前向结果


## 与@torch.enable_grad()配合

In [5]:
def enable_grad_demo():
    '''
    在no_grad上下文中临时启用梯度
    '''
    print("=== 临时启用梯度 ===")
    x = torch.tensor([1.0],requires_grad = True)
    
    # 全局禁用梯度
    with torch.no_grad():
        print('在no_grad上下文中：')
        y1 = x ** 2
        print(f"y1.requires_grad：{y1.requires_grad}")
        
        # 临时启用梯度
        y2 = x ** 2
        print(f"y2.requires_grad：{y2.requires_grad}")
        print(f"y2.grad_fn：{y2.grad_fn}")
        
    # 实际应用：计算梯度但不需要保存中间结果
    print("\n实际应用 - 计算梯度但优化内存:")
    
    def compute_gradient_efficiently(model,x,y):
        '''
        高效计算梯度：前向传播不保存中间结果
        '''
        with torch.no_grad():
            output = model(x)
            
        # 启用梯度重新计算loss
        with torch.enable_grad():
            output.requires_grad_(True) # 这里使用的原地操作
            loss = F.mse_loss(output,y)
            loss.backward()
        return loss
    
    model = nn.Linear(10, 5)
    x = torch.randn(32, 10)
    y = torch.randn(32, 5)
    
    loss = compute_gradient_efficiently(model, x, y)
    print(f"Loss: {loss.item():.4f}")
    print(f"梯度存在: {model.weight.grad is not None}")

enable_grad_demo()

=== 临时启用梯度 ===
在no_grad上下文中：
y1.requires_grad：False
y2.requires_grad：False
y2.grad_fn：None

实际应用 - 计算梯度但优化内存:
Loss: 1.2087
梯度存在: False


## 性能对比

In [2]:
def performance_comparison():
    """
    性能对比：有无 no_grad
    """
    print("=== 性能对比 ===\n")
    
    class DeepModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.layers = nn.Sequential(
                *[nn.Linear(1000, 1000) for _ in range(10)]
            )
        
        def forward(self, x):
            return self.layers(x)
    
    model = DeepModel()
    x = torch.randn(100, 1000)
    
    import time
    
    # 测试1：前向传播（有梯度）
    start = time.time()
    for _ in range(100):
        output = model(x)
    time_with_grad = time.time() - start
    
    # 测试2：前向传播（无梯度）
    start = time.time()
    with torch.no_grad():
        for _ in range(100):
            output = model(x)
    time_no_grad = time.time() - start
    
    print(f"有梯度推理: {time_with_grad:.4f}s")
    print(f"无梯度推理: {time_no_grad:.4f}s")
    print(f"加速比: {time_with_grad/time_no_grad:.2f}x")
    
    # 内存对比
    import tracemalloc
    
    # 有梯度
    tracemalloc.start()
    output = model(x)
    current, peak_with = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    # 无梯度
    tracemalloc.start()
    with torch.no_grad():
        output = model(x)
    current, peak_without = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    print(f"\n有梯度峰值内存: {peak_with / 1024**2:.2f} MB")
    print(f"无梯度峰值内存: {peak_without / 1024**2:.2f} MB")
    print(f"内存节省: {(peak_with - peak_without) / 1024**2:.2f} MB")

performance_comparison()

=== 性能对比 ===

有梯度推理: 1.8016s
无梯度推理: 2.0089s
加速比: 0.90x

有梯度峰值内存: 0.00 MB
无梯度峰值内存: 0.00 MB
内存节省: 0.00 MB


## ⚠️ 常见陷阱与注意事项

In [2]:
def common_pitfalls():
    """
    常见错误和注意事项
    """
    print("=== 常见陷阱 ===\n")
    
    # === 陷阱1：在需要梯度的地方使用 no_grad ===
    print("陷阱1: 错误使用 no_grad 导致无法计算梯度")
    
    x = torch.tensor([1.0], requires_grad=True)
    
    # 错误：在 no_grad 中计算loss
    with torch.no_grad():
        loss = (x - 2) ** 2
    try:
        loss.backward()
    except RuntimeError as e:
        print(f"  错误: {e}")
    
    # 正确：不在 no_grad 中计算loss
    loss = (x - 2) ** 2
    loss.backward()
    print(f"  正确: x.grad = {x.grad.item()}")
    
    # === 陷阱2：影响 in-place 操作 === 
    print("\n陷阱2: no_grad 不影响 in-place 操作")
    
    x = torch.tensor([1.0], requires_grad=True)
    
    with torch.no_grad():
        x.mul_(2)  # in-place 修改
    
    print(f"  x: {x.item()}, requires_grad={x.requires_grad}")
    print(f"  x.grad_fn: {x.grad_fn}")
    
    # === 陷阱3：与 model.eval() 的区别 === 
    print("\n陷阱3: no_grad vs model.eval()")
    
    model = nn.Dropout(0.5)
    x = torch.ones(10)
    
    # model.eval() 影响特定层的行为
    model.eval()
    with torch.no_grad():
        output1 = model(x)
    
    model.train()
    with torch.no_grad():
        output2 = model(x)
    
    print(f"  eval模式输出: {output1}")
    print(f"  train模式输出: {output2}")
    print(f"  差异: {torch.allclose(output1, output2)}")  # Dropout行为不同
    
    # === 陷阱4：梯度累积的影响 === 
    print("\n陷阱4: no_grad 不累积梯度")
    
    x = torch.tensor([1.0], requires_grad=True)
    
    for i in range(3):
        with torch.no_grad():
            y = x * 2
        # y 没有梯度，无法反向传播
    
    print("  梯度无法累积，因为 y 没有 requires_grad")

common_pitfalls()

=== 常见陷阱 ===

陷阱1: 错误使用 no_grad 导致无法计算梯度
  错误: element 0 of tensors does not require grad and does not have a grad_fn
  正确: x.grad = -2.0

陷阱2: no_grad 不影响 in-place 操作
  x: 2.0, requires_grad=True
  x.grad_fn: None

陷阱3: no_grad vs model.eval()
  eval模式输出: tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
  train模式输出: tensor([2., 0., 2., 2., 2., 0., 2., 0., 2., 0.])
  差异: False

陷阱4: no_grad 不累积梯度
  梯度无法累积，因为 y 没有 requires_grad


## 最佳实践

In [3]:
def best_practices():
    """
    最佳实践总结
    """
    print("=== 最佳实践 ===\n")
    
    practices = {
        "推理/验证": """
            @torch.no_grad()
            def evaluate(model, dataloader):
                model.eval()
                for batch in dataloader:
                    output = model(batch)
                    # 计算指标...
        """,
        
        "参数更新": """
            @torch.no_grad()
            def update_parameters(optimizer):
                for param in optimizer.param_groups[0]['params']:
                    param.data -= lr * param.grad
        """,
        
        "梯度检查点": """
            def checkpoint_forward(module, *inputs):
                with torch.no_grad():
                    outputs = module(*inputs)
                return outputs
        """,
        
        "混合使用": """
            # 禁用梯度但保持训练模式
            with torch.no_grad():
                # 计算某些统计量
                stats = compute_statistics(model, x)
            
            # 继续训练
            loss = compute_loss(model, x, y)
            loss.backward()
        """,
        
        "性能优化": """
            # 大批量推理时使用
            with torch.no_grad():
                for batch in dataloader:
                    outputs = model(batch)
                    # 收集结果
        """
    }
    
    for name, code in practices.items():
        print(f"\n{name}:")
        print(code)
    
    print("\n关键要点:")
    print("• 推理/验证时总是使用 no_grad")
    print("• 优化器内部使用 no_grad 更新参数")
    print("• no_grad 不影响模型模式（train/eval）")
    print("• 与 model.eval() 配合使用获得最佳效果")
    print("• 使用装饰器简化代码")

best_practices()

=== 最佳实践 ===


推理/验证:

            @torch.no_grad()
            def evaluate(model, dataloader):
                model.eval()
                for batch in dataloader:
                    output = model(batch)
                    # 计算指标...
        

参数更新:

            @torch.no_grad()
            def update_parameters(optimizer):
                for param in optimizer.param_groups[0]['params']:
                    param.data -= lr * param.grad
        

梯度检查点:

            def checkpoint_forward(module, *inputs):
                with torch.no_grad():
                    outputs = module(*inputs)
                return outputs
        

混合使用:

            # 禁用梯度但保持训练模式
            with torch.no_grad():
                # 计算某些统计量
                stats = compute_statistics(model, x)
            
            # 继续训练
            loss = compute_loss(model, x, y)
            loss.backward()
        

性能优化:

            # 大批量推理时使用
            with torch.no_grad():
                for batch in dat